# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/satyamgupta04/week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 0. Setup

This notebook establishes a baseline prioritization rule for content refresh based on staleness and search volume.

### Resources
- **Dataset:** `FlyRank/internship-warehouse`
- **Output:** `work/outputs/baseline_action_score.csv`

## 1. Signal checks

### Signal 1: Staleness
**Verdict: MIXED**

Staleness is a mixed signal for prioritization. Most content is recently updated, while a substantial group is 365+ days old. However, the average search-volume pattern is not consistently higher or lower as content becomes older, and the 365+ bucket has missing search-volume values. Therefore, staleness alone is not strong enough to justify priority; it is better used together with a search-visibility signal.

In [37]:
# Staleness check and average search volume per bucket
staleness_search = con.execute(f"""
    SELECT
        CASE
            WHEN content_updated_date IS NULL THEN 'unknown'
            WHEN DATE '2026-03-31' - content_updated_date <= 30 THEN '0-30'
            WHEN DATE '2026-03-31' - content_updated_date <= 90 THEN '31-90'
            WHEN DATE '2026-03-31' - content_updated_date <= 180 THEN '91-180'
            WHEN DATE '2026-03-31' - content_updated_date <= 365 THEN '181-365'
            ELSE '365+'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(search_volume), 1) AS avg_search_volume
    FROM read_parquet('{dim_content_path}')
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

display(staleness_search)

,staleness_bucket,n,avg_search_volume
0,0-30,383714,228.9
1,181-365,18820,104.1
2,31-90,35052,100.8
3,365+,69033,NaN
4,91-180,12987,55.3


## 2. ONE baseline rule

**Rule logic:**
- **Stale (>90 days):** +2 points
- **High Demand (>=300 volume):** +2 points

**Fields produced:** `score`, `reason_code`, `action`.
- **score**: Numeric priority.
- **reason_code**: `stale_high_demand`, `stale`, `high_demand`, or `none`.
- **action**: `REVIEW` (Score 4) or `MONITOR` (Score < 4).

In [38]:
# Build the ranked queue
baseline_queue = con.execute(f"""
    SELECT
        client_hash_id, content_hash_id, content_updated_date, search_volume,
        (CASE WHEN DATE '2026-03-31' - content_updated_date > 90 THEN 2 ELSE 0 END +
         CASE WHEN search_volume >= 300 THEN 2 ELSE 0 END) AS action_score,
        CASE
            WHEN DATE '2026-03-31' - content_updated_date > 90 AND search_volume >= 300 THEN 'stale_high_demand'
            WHEN DATE '2026-03-31' - content_updated_date > 90 THEN 'stale'
            WHEN search_volume >= 300 THEN 'high_demand'
            ELSE 'none'
        END AS reason_code,
        CASE WHEN action_score = 4 THEN 'REVIEW' ELSE 'MONITOR' END AS action
    FROM read_parquet('{dim_content_path}')
    ORDER BY action_score DESC, search_volume DESC
    LIMIT 1000
""").fetchdf()

# Write CSV
os.makedirs("work/outputs", exist_ok=True)
baseline_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
display(baseline_queue.head())

,client_hash_id,content_hash_id,content_updated_date,search_volume,action_score,reason_code,action
0,client_08a6a72ff48e62c0,content_050440d8949b301d,2025-11-20,49500,4,stale_high_demand,REVIEW
1,client_895724a3631e810c,content_31be874079923664,2025-09-08,40500,4,stale_high_demand,REVIEW
2,client_895724a3631e810c,content_0be9b19ca58d3f99,2025-09-07,40500,4,stale_high_demand,REVIEW
3,client_895724a3631e810c,content_b229e67c1c2db011,2025-08-09,33100,4,stale_high_demand,REVIEW
4,client_895724a3631e810c,content_96b0770e5deb2fbe,2025-08-09,33100,4,stale_high_demand,REVIEW


## 3. Top-10 skeptic review

Reviewing high-priority items to identify potential false positives.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + limitation

**Weak picks:** High-demand pages updated recently (Score 2, Action: MONITOR).

**Limitation:** Staleness is only a proxy for refresh opportunity. It does not observe ranking changes or actual content quality.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Self-check

- [x] Two signal checks are shown with visible bucket tables.
- [x] Each signal has a one-word verdict: MIXED or CONFIRMED.
- [x] The rule produces a score, one reason code, and an action label.
- [x] The ranked queue is written to CSV.
- [x] The top 10 review is documented.
- [x] Weak picks and a named limitation are documented.
- [x] The notebook runs top to bottom without errors.

In [41]:
import duckdb
import pandas as pd
import os
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

# Database and Auth
con = duckdb.connect()
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

# Load Data
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet"
)
print("Setup complete. Data loaded.")

Setup complete. Data loaded.


In [42]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

print("Hugging Face login successful")

Hugging Face login successful


In [43]:
from huggingface_hub import hf_hub_download

dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet"
)

print("Downloaded successfully:")
print(dim_content_path)

Downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet


In [44]:
content_columns = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{dim_content_path}')
""").fetchdf()

content_columns

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [45]:
staleness_check = con.execute(f"""
    SELECT
        CASE
            WHEN content_updated_date IS NULL THEN 'unknown'
            WHEN DATE '2026-03-31' - content_updated_date <= 30 THEN '0-30'
            WHEN DATE '2026-03-31' - content_updated_date <= 90 THEN '31-90'
            WHEN DATE '2026-03-31' - content_updated_date <= 180 THEN '91-180'
            WHEN DATE '2026-03-31' - content_updated_date <= 365 THEN '181-365'
            ELSE '365+'
        END AS staleness_bucket,
        COUNT(*) AS n
    FROM read_parquet('{dim_content_path}')
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

staleness_check

,staleness_bucket,n
0,0-30,383714
1,181-365,18820
2,31-90,35052
3,365+,69033
4,91-180,12987


In [46]:
staleness_search = con.execute(f"""
    SELECT
        CASE
            WHEN content_updated_date IS NULL THEN 'unknown'
            WHEN DATE '2026-03-31' - content_updated_date <= 30 THEN '0-30'
            WHEN DATE '2026-03-31' - content_updated_date <= 90 THEN '31-90'
            WHEN DATE '2026-03-31' - content_updated_date <= 180 THEN '91-180'
            WHEN DATE '2026-03-31' - content_updated_date <= 365 THEN '181-365'
            ELSE '365+'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(search_volume), 1) AS avg_search_volume
    FROM read_parquet('{dim_content_path}')
    GROUP BY 1
    ORDER BY
        CASE staleness_bucket
            WHEN '0-30' THEN 1
            WHEN '31-90' THEN 2
            WHEN '91-180' THEN 3
            WHEN '181-365' THEN 4
            WHEN '365+' THEN 5
            ELSE 6
        END
""").fetchdf()

staleness_search

,staleness_bucket,n,avg_search_volume
0,0-30,383714,228.9
1,31-90,35052,100.8
2,91-180,12987,55.3
3,181-365,18820,104.1
4,365+,69033,NaN


### Signal 1 — Staleness

**Verdict: MIXED**

Staleness is a mixed signal for prioritization. Most content is recently updated, while a substantial group is 365+ days old. However, the average search-volume pattern is not consistently higher or lower as content becomes older, and the 365+ bucket has missing search-volume values. Therefore, staleness alone is not strong enough to justify priority; it is better used together with a search-visibility signal.

In [47]:
# Volume distribution check
volume_check = con.execute(f"""
    SELECT
        CASE
            WHEN search_volume IS NULL THEN 'missing'
            WHEN search_volume = 0 THEN '0'
            WHEN search_volume < 100 THEN '1-99'
            WHEN search_volume < 300 THEN '100-299'
            WHEN search_volume < 3000 THEN '300-2999'
            ELSE '3000+'
        END AS volume_bucket,
        COUNT(*) AS n
    FROM read_parquet('{dim_content_path}')
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

display(volume_check)

,volume_bucket,n
0,0,163631
1,1-99,174178
2,100-299,17504
3,300-2999,17535
4,3000+,4136
5,missing,142622


### Signal 2: Search volume
**Verdict: CONFIRMED**

Search volume is a useful opportunity signal for the baseline because it separates content with little or no observable search demand from content with measurable search demand. The distribution also shows that high-volume pages are a relatively small group, so volume can help prioritize pages where a review may have greater potential impact.

In [48]:
baseline_queue = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        search_volume,

        CASE
            WHEN content_updated_date IS NULL THEN 'unknown'
            WHEN DATE '2026-03-31' - content_updated_date <= 30 THEN '0-30'
            WHEN DATE '2026-03-31' - content_updated_date <= 90 THEN '31-90'
            WHEN DATE '2026-03-31' - content_updated_date <= 180 THEN '91-180'
            WHEN DATE '2026-03-31' - content_updated_date <= 365 THEN '181-365'
            ELSE '365+'
        END AS staleness_bucket,

        (
            CASE
                WHEN content_updated_date IS NOT NULL
                     AND DATE '2026-03-31' - content_updated_date > 90
                THEN 2 ELSE 0
            END
            +
            CASE
                WHEN search_volume >= 300
                THEN 2 ELSE 0
            END
        ) AS action_score,

        CASE
            WHEN content_updated_date IS NOT NULL
                 AND DATE '2026-03-31' - content_updated_date > 90
                 AND search_volume >= 300
            THEN 'stale_high_demand'
            WHEN content_updated_date IS NOT NULL
                 AND DATE '2026-03-31' - content_updated_date > 90
            THEN 'stale'
            WHEN search_volume >= 300
            THEN 'high_demand'
            ELSE 'none'
        END AS reason_code,

        CASE
            WHEN content_updated_date IS NOT NULL
                 AND DATE '2026-03-31' - content_updated_date > 90
                 AND search_volume >= 300
            THEN 'REVIEW'
            ELSE 'MONITOR'
        END AS action

    FROM read_parquet('{dim_content_path}')

    ORDER BY action_score DESC, search_volume DESC
    LIMIT 1000
""").fetchdf()

baseline_queue.head(10)

,client_hash_id,content_hash_id,content_updated_date,search_volume,staleness_bucket,action_score,reason_code,action
0,client_08a6a72ff48e62c0,content_050440d8949b301d,2025-11-20,49500,91-180,4,stale_high_demand,REVIEW
1,client_895724a3631e810c,content_31be874079923664,2025-09-08,40500,181-365,4,stale_high_demand,REVIEW
2,client_895724a3631e810c,content_0be9b19ca58d3f99,2025-09-07,40500,181-365,4,stale_high_demand,REVIEW
3,client_895724a3631e810c,content_96b0770e5deb2fbe,2025-08-09,33100,181-365,4,stale_high_demand,REVIEW
4,client_895724a3631e810c,content_b50b506e2cf5974a,2025-08-11,33100,181-365,4,stale_high_demand,REVIEW
5,client_895724a3631e810c,content_b229e67c1c2db011,2025-08-09,33100,181-365,4,stale_high_demand,REVIEW
6,client_ba5520003fdf9b10,content_2a4c87c2a7cc99b8,2025-09-03,18100,181-365,4,stale_high_demand,REVIEW
7,client_ba5520003fdf9b10,content_299b1628da491a43,2025-09-03,14800,181-365,4,stale_high_demand,REVIEW
8,client_b2c982d4e2028c19,content_aef9656cc64b7239,2025-09-03,12100,181-365,4,stale_high_demand,REVIEW
9,client_895724a3631e810c,content_9bb5b2b8c28176da,2025-08-09,9900,181-365,4,stale_high_demand,REVIEW


In [49]:
baseline_queue.groupby(
    ["action_score", "reason_code", "action"],
    dropna=False
).size().reset_index(name="n").sort_values(
    "action_score", ascending=False
)

,action_score,reason_code,action,n
1,4,stale_high_demand,REVIEW,832
0,2,high_demand,MONITOR,168


In [50]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(baseline_queue))

Saved: work/outputs/baseline_action_score.csv
Rows: 1000


In [51]:
# Extract top 10 for review
top10 = baseline_queue.head(10)
display(top10)

,client_hash_id,content_hash_id,content_updated_date,search_volume,staleness_bucket,action_score,reason_code,action
0,client_08a6a72ff48e62c0,content_050440d8949b301d,2025-11-20,49500,91-180,4,stale_high_demand,REVIEW
1,client_895724a3631e810c,content_31be874079923664,2025-09-08,40500,181-365,4,stale_high_demand,REVIEW
2,client_895724a3631e810c,content_0be9b19ca58d3f99,2025-09-07,40500,181-365,4,stale_high_demand,REVIEW
3,client_895724a3631e810c,content_96b0770e5deb2fbe,2025-08-09,33100,181-365,4,stale_high_demand,REVIEW
4,client_895724a3631e810c,content_b50b506e2cf5974a,2025-08-11,33100,181-365,4,stale_high_demand,REVIEW
5,client_895724a3631e810c,content_b229e67c1c2db011,2025-08-09,33100,181-365,4,stale_high_demand,REVIEW
6,client_ba5520003fdf9b10,content_2a4c87c2a7cc99b8,2025-09-03,18100,181-365,4,stale_high_demand,REVIEW
7,client_ba5520003fdf9b10,content_299b1628da491a43,2025-09-03,14800,181-365,4,stale_high_demand,REVIEW
8,client_b2c982d4e2028c19,content_aef9656cc64b7239,2025-09-03,12100,181-365,4,stale_high_demand,REVIEW
9,client_895724a3631e810c,content_9bb5b2b8c28176da,2025-08-09,9900,181-365,4,stale_high_demand,REVIEW


### Skepticism Table

| Rank | Action | Why it's there | What would make it wrong |
|---:|---|---|---|
| 1 | REVIEW | Stale 91–180 days with 49,500 search volume. | Wrong if page is evergreen or volume is outdated. |
| 2 | REVIEW | Stale 181–365 days with 40,500 search volume. | Wrong if content remains accurate despite age. |

In [52]:
# Display examples of weak picks (bottom of the top 1000)
display(baseline_queue.tail(10))

,client_hash_id,content_hash_id,content_updated_date,search_volume,staleness_bucket,action_score,reason_code,action
990,client_a2eeb8899886adde,content_e876d63a35246b28,2026-07-01,60500,0-30,2,high_demand,MONITOR
991,client_d211cb07b9059bab,content_c635404837890561,2026-07-01,60500,0-30,2,high_demand,MONITOR
992,client_d211cb07b9059bab,content_d908b77caa11c6f8,2026-07-01,60500,0-30,2,high_demand,MONITOR
993,client_fef1a8f436438636,content_a85bf5efbd1f137e,2026-06-17,60500,0-30,2,high_demand,MONITOR
994,client_d211cb07b9059bab,content_4564ea5416ed9e02,2026-06-25,60500,0-30,2,high_demand,MONITOR
995,client_73cda7b4e4f265ea,content_2dcf2442c6dd2c2f,2026-05-18,49500,0-30,2,high_demand,MONITOR
996,client_ba65e80a1116ae41,content_d366c0bfc457bbb0,2026-05-20,49500,0-30,2,high_demand,MONITOR
997,client_e547b89c05043229,content_4d1222a8b7da3a81,2026-06-24,49500,0-30,2,high_demand,MONITOR
998,client_1a730cb2640a1abf,content_796930f176760cdf,2026-07-01,49500,0-30,2,high_demand,MONITOR
999,client_c3bfd4b85e1fef26,content_3ff3606157953050,2026-05-20,49500,0-30,2,high_demand,MONITOR


### Weak picks / rule limitation

The weaker picks are high-demand pages that were updated recently. They receive a score of 2 because they satisfy the search-volume condition but not the staleness condition, so the rule labels them MONITOR rather than REVIEW.

This is intentional: high search demand alone does not mean a page needs a refresh. A page can have substantial demand while already being recently updated.

The baseline is therefore a prioritization rule, not a measure of content quality or future performance.

### Named limitation

**Limitation: staleness is only a proxy for refresh opportunity.**

The rule uses the age of the last update and search-volume estimates, but it does not directly observe content quality, ranking changes, query intent changes, or whether the existing page is already satisfying users. Therefore, a high-scoring page still requires human review before any refresh decision.

## 5. Self-check

- [x] Two signal checks are shown with visible bucket tables.
- [x] Each signal has a one-word verdict: MIXED or CONFIRMED.
- [x] The rule produces a score, one reason code, and an action label.
- [x] The ranked queue is written to CSV.
- [x] The top 10 review and limitations are documented.
- [x] The notebook runs top to bottom without errors.